Imports library and data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent  
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

customer_path = str(RAW_DIR / "customer_info.parquet")
usage_path = str(RAW_DIR / "usage.parquet")
calls_path = str(RAW_DIR / "calls.csv")
cease_path = str(RAW_DIR / "cease.csv")

RAW_DIR

WindowsPath('c:/Users/icons/Documents/UKTelecomCeasePrediction/data/raw')

File sizes check

In [2]:
for p in RAW_DIR.iterdir():
    print(f"{p.name:22s} {p.stat().st_size/1e9:8.3f} GB")

calls.csv                 0.059 GB
cease.csv                 0.018 GB
customer_info.parquet     0.270 GB
usage.parquet             3.094 GB


Load tables

In [3]:
cease = pd.read_csv(
    cease_path,
    parse_dates=["cease_placed_date", "cease_completed_date"],
)

calls = pd.read_csv(
    calls_path,
    usecols=["unique_customer_identifier", "event_date", "call_type",
             "talk_time_seconds", "hold_time_seconds"],
    parse_dates=["event_date"],
    dtype={"call_type": "category"},
)

print("cease:", cease.shape, "calls:", calls.shape)
display(cease.head(3))
display(calls.head(3))

cease: (146363, 5) calls: (628437, 5)


,unique_customer_identifier,cease_placed_date,cease_completed_date,reason_description,reason_description_insight
0,03b1c584533a86d067dd51bbca242db2b55b692f10d325...,2023-08-03,2023-09-04,Competitor Deals - No longer required,CompetitorDeals
1,97a7bdce317de91a32636e6675bbb2e5b25573308ef7bb...,2023-08-03,2023-09-04,Cease,VagueReason
2,c5049a1aedc36d7d7379c2c2144972b099521e6614cf8c...,2023-08-03,2023-09-05,Competitor Deals - No longer required,CompetitorDeals


,unique_customer_identifier,event_date,call_type,talk_time_seconds,hold_time_seconds
0,aae0258b41e6e88365d7d5ce648ea69d837602b4bb419e...,2023-02-22,Loyalty,627.0,235.0
1,15f9f6fc1872bbf6963a84de253d600e5d18d75d7784ce...,2023-03-16,Tech,267.0,293.0
2,c18d59888cb050a5694d1e613a277d79b4a3083bd1b813...,2023-02-22,Loyalty,689.0,0.0


Checking datatypes and missing data

In [4]:
def missing_report(df, name):
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    print(f"\n--- {name} dtypes ---")
    print(df.dtypes)
    print(f"\n--- {name} missing % (top 15) ---")
    display(miss.head(15))

missing_report(cease, "cease")
missing_report(calls, "calls")


--- cease dtypes ---
unique_customer_identifier            object
cease_placed_date             datetime64[ns]
cease_completed_date          datetime64[ns]
reason_description                    object
reason_description_insight            object
dtype: object

--- cease missing % (top 15) ---


cease_completed_date          18.595547
unique_customer_identifier     0.000000
cease_placed_date              0.000000
reason_description             0.000000
reason_description_insight     0.000000
dtype: float64


--- calls dtypes ---
unique_customer_identifier            object
event_date                    datetime64[ns]
call_type                           category
talk_time_seconds                    float64
hold_time_seconds                    float64
dtype: object

--- calls missing % (top 15) ---


call_type                     2.855974
unique_customer_identifier    0.000000
event_date                    0.000000
talk_time_seconds             0.000000
hold_time_seconds             0.000000
dtype: float64

Print date ranges

In [5]:
print("cease placed range:", cease["cease_placed_date"].min(), "->", cease["cease_placed_date"].max())
print("calls event range:", calls["event_date"].min(), "->", calls["event_date"].max())

cease placed range: 2022-08-01 00:00:00 -> 2024-09-01 00:00:00
calls event range: 2022-08-01 00:00:00 -> 2024-09-01 00:00:00


DuckDB connect for big tables

In [6]:
con = duckdb.connect()

customer_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{customer_path}')"
).fetchdf()

usage_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{usage_path}')"
).fetchdf()

print("customer_info columns:", len(customer_cols))
print("usage columns:", len(usage_cols))

display(customer_cols.head(40))
display(usage_cols.head(40))

customer_info columns: 12
usage columns: 4


,column_name,column_type,null,key,default,extra
0,unique_customer_identifier,VARCHAR,YES,None,None,None
1,datevalue,DATE,YES,None,None,None
2,contract_status,VARCHAR,YES,None,None,None
3,contract_dd_cancels,BIGINT,YES,None,None,None
4,dd_cancel_60_day,INTEGER,YES,None,None,None
5,ooc_days,INTEGER,YES,None,None,None
6,technology,VARCHAR,YES,None,None,None
7,speed,INTEGER,YES,None,None,None
8,line_speed,DOUBLE,YES,None,None,None
9,sales_channel,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,unique_customer_identifier,VARCHAR,YES,None,None,None
1,calendar_date,DATE,YES,None,None,None
2,usage_download_mbs,VARCHAR,YES,None,None,None
3,usage_upload_mbs,VARCHAR,YES,None,None,None


In [7]:
display(customer_cols["column_name"])
display(usage_cols["column_name"])

0     unique_customer_identifier
1                      datevalue
2                contract_status
3            contract_dd_cancels
4               dd_cancel_60_day
5                       ooc_days
6                     technology
7                          speed
8                     line_speed
9                  sales_channel
10              crm_package_name
11                   tenure_days
Name: column_name, dtype: object

0    unique_customer_identifier
1                 calendar_date
2            usage_download_mbs
3              usage_upload_mbs
Name: column_name, dtype: object

Customer snapshot - dates

In [8]:
snap_sample = con.execute(f"""
SELECT DISTINCT datevalue
FROM read_parquet('{customer_path}')
ORDER BY datevalue
LIMIT 15
""").fetchdf()

snap_sample

,datevalue
0,2022-08-01
1,2022-09-01
2,2022-10-01
3,2022-11-01
4,2022-12-01
5,2023-01-01
6,2023-02-01
7,2023-03-01
8,2023-04-01
9,2023-05-01


Fix call_type missing data

In [9]:
calls["call_type"] = calls["call_type"].cat.add_categories(["Unknown"]).fillna("Unknown")
calls["call_type"].value_counts(dropna=False).head(10)

call_type
Tech                171942
CS&B                167117
Loyalty             144269
Customer Finance     71105
FTTP                 43926
Unknown              17948
Other                 5385
Order Management      4422
Complaints            1504
TTB - Sales            369
Name: count, dtype: int64

Register pandas tables into DuckDB

In [10]:
con.register("calls_pd", calls)
con.register("cease_pd", cease)

Defining target and feature windows

Predicting if a Customer will place a cease within the nexr 30 days

Set Windows

In [11]:
LOOKBACK_DAYS = 30
HORIZON_DAYS = 30

Building the modeling table with DuckDB

In [12]:
out_path = str(INTERIM_DIR / "modeling_table.parquet")

con.execute(f"""
COPY (
WITH base AS (
    SELECT
        unique_customer_identifier,
        CAST(datevalue AS DATE) AS snapshot_date,
        contract_status,
        contract_dd_cancels,
        dd_cancel_60_day,
        ooc_days,
        technology,
        speed,
        line_speed,
        sales_channel,
        crm_package_name,
        tenure_days
    FROM read_parquet('{customer_path}')
),

labels AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        CASE WHEN EXISTS (
            SELECT 1
            FROM cease_pd c
            WHERE c.unique_customer_identifier = b.unique_customer_identifier
              AND CAST(c.cease_placed_date AS DATE) > b.snapshot_date
              AND CAST(c.cease_placed_date AS DATE) <= b.snapshot_date + INTERVAL '{HORIZON_DAYS} days'
        ) THEN 1 ELSE 0 END AS target_30d
    FROM base b
),

calls_agg AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        COUNT(ca.event_date) AS calls_30d,
        AVG(ca.talk_time_seconds) AS avg_talk_time_30d,
        AVG(ca.hold_time_seconds) AS avg_hold_time_30d,
        SUM(CASE WHEN ca.call_type = 'Loyalty' THEN 1 ELSE 0 END) AS loyalty_calls_30d,
        SUM(CASE WHEN ca.call_type = 'Tech' THEN 1 ELSE 0 END) AS tech_calls_30d,
        SUM(CASE WHEN ca.call_type = 'Billing' THEN 1 ELSE 0 END) AS billing_calls_30d,
        SUM(CASE WHEN ca.call_type = 'Unknown' THEN 1 ELSE 0 END) AS unknown_calls_30d
    FROM base b
    LEFT JOIN calls_pd ca
      ON ca.unique_customer_identifier = b.unique_customer_identifier
     AND CAST(ca.event_date AS DATE) > b.snapshot_date - INTERVAL '{LOOKBACK_DAYS} days'
     AND CAST(ca.event_date AS DATE) <= b.snapshot_date
    GROUP BY 1,2
),

usage_agg AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        AVG(TRY_CAST(u.usage_download_mbs AS DOUBLE)) AS avg_download_30d,
        AVG(TRY_CAST(u.usage_upload_mbs   AS DOUBLE)) AS avg_upload_30d,
        SUM(TRY_CAST(u.usage_download_mbs AS DOUBLE)) AS sum_download_30d,
        SUM(TRY_CAST(u.usage_upload_mbs   AS DOUBLE)) AS sum_upload_30d
    FROM base b
    LEFT JOIN read_parquet('{usage_path}') u
      ON u.unique_customer_identifier = b.unique_customer_identifier
     AND CAST(u.calendar_date AS DATE) > b.snapshot_date - INTERVAL '{LOOKBACK_DAYS} days'
     AND CAST(u.calendar_date AS DATE) <= b.snapshot_date
    GROUP BY 1,2
)

SELECT
    b.*,
    l.target_30d,
    ca.calls_30d, ca.avg_talk_time_30d, ca.avg_hold_time_30d,
    ca.loyalty_calls_30d, ca.tech_calls_30d, ca.billing_calls_30d, ca.unknown_calls_30d,
    ua.avg_download_30d, ua.avg_upload_30d, ua.sum_download_30d, ua.sum_upload_30d
FROM base b
JOIN labels l
  ON l.unique_customer_identifier = b.unique_customer_identifier
 AND l.snapshot_date = b.snapshot_date
LEFT JOIN calls_agg ca
  ON ca.unique_customer_identifier = b.unique_customer_identifier
 AND ca.snapshot_date = b.snapshot_date
LEFT JOIN usage_agg ua
  ON ua.unique_customer_identifier = b.unique_customer_identifier
 AND ua.snapshot_date = b.snapshot_date
) TO '{out_path}' (FORMAT PARQUET);
""")

print("Saved:", out_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\data\interim\modeling_table.parquet


Validate target distribution

In [13]:
modeling_path = str(INTERIM_DIR / "modeling_table.parquet")

target_stats = con.execute(f"""
SELECT target_30d, COUNT(*) AS n
FROM read_parquet('{modeling_path}')
GROUP BY target_30d
ORDER BY target_30d
""").fetchdf()

target_stats

,target_30d,n
0,0,3431906
1,1,139268


Focusing on PR-AUC, Recall/Precision trade-off

Confirm positive rate and class weights

In [14]:
total = int(target_stats["n"].sum())
pos = int(target_stats.loc[target_stats["target_30d"] == 1, "n"].iloc[0])
neg = total - pos

pos_rate = pos / total
neg_rate = neg / total

print("total:", total, "pos:", pos, "neg:", neg)
print("pos_rate:", pos_rate, "neg_rate:", neg_rate)

# Common weight ratio (rough idea): how many negatives per positive
print("neg/pos ratio:", neg / pos)

total: 3571174 pos: 139268 neg: 3431906
pos_rate: 0.03899781976459282 neg_rate: 0.9610021802354072
neg/pos ratio: 24.642459143521844


Checking missing data

In [16]:
missing_check = con.execute(f"""
SELECT
  AVG(CASE WHEN calls_30d IS NULL THEN 1 ELSE 0 END) * 100 AS pct_calls30d_null,
  AVG(CASE WHEN avg_download_30d IS NULL THEN 1 ELSE 0 END) * 100 AS pct_avgdl_null,
  AVG(CASE WHEN contract_status IS NULL THEN 1 ELSE 0 END) * 100 AS pct_contractstatus_null,
  AVG(CASE WHEN technology IS NULL THEN 1 ELSE 0 END) * 100 AS pct_tech_null
FROM read_parquet('{modeling_path}')
""").fetchdf()

missing_check

,pct_calls30d_null,pct_avgdl_null,pct_contractstatus_null,pct_tech_null
0,0.0,11.328992,0.0,0.0


Count duplicates in the modeling table

In [22]:
modeling_path = str(INTERIM_DIR / "modeling_table.parquet")

dup_summary = con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_unique_keys,
  COUNT(*) - COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_duplicates
FROM read_parquet('{modeling_path}')
""").fetchdf()

dup_summary

,n_rows,n_unique_keys,n_duplicates
0,3571174,3532720,38454


Show top duplicate keys

In [23]:
dup_keys = con.execute(f"""
SELECT unique_customer_identifier, snapshot_date, COUNT(*) AS cnt
FROM read_parquet('{modeling_path}')
GROUP BY 1,2
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20
""").fetchdf()

dup_keys

,unique_customer_identifier,snapshot_date,cnt
0,6ae8dc33bdb1bb8a8a3d36f4890963b26722e4211985d4...,2024-07-01,4
1,02a8e5a89a690e7d4d293239000e9939a18987eee9fc9f...,2022-12-01,4
2,2f63e6d216d89ea51cc14ee6d7bd0767e79d3d0a9d0699...,2023-05-01,4
3,d2432b0c8402cfe0c1a17b5cbf1051f5193e0958bebb6e...,2023-03-01,4
4,8231ad7bcfb81b1abb7ddde4ca2abf71ac7ea128d0af92...,2024-04-01,4
5,4d91e47e796951dc64a5c16c5feee81304ddbf70a9fee7...,2023-08-01,4
6,5b5ec9711cc240be2b0fdc85e8908ce87774af385c7593...,2023-11-01,4
7,f76d80b54b528d36f01f35b7260e70d1c2fabb5454f976...,2024-01-01,4
8,83d5ce1b6e543e66eba1fd3119a372f330b709e17c8354...,2023-10-01,4
9,99c1ef90239f1a6811e3bfef7a0f28578ae296dc8d864e...,2023-05-01,4


Create deduped modeling table

In [24]:
dedup_path = str(INTERIM_DIR / "modeling_table_dedup.parquet")

con.execute(f"""
COPY (
  SELECT *
  FROM (
    SELECT
      *,
      -- null_score: fewer nulls = better
      (
        CASE WHEN contract_status IS NULL THEN 1 ELSE 0 END +
        CASE WHEN technology IS NULL THEN 1 ELSE 0 END +
        CASE WHEN sales_channel IS NULL THEN 1 ELSE 0 END +
        CASE WHEN crm_package_name IS NULL THEN 1 ELSE 0 END +
        CASE WHEN calls_30d IS NULL THEN 1 ELSE 0 END +
        CASE WHEN avg_download_30d IS NULL THEN 1 ELSE 0 END +
        CASE WHEN sum_download_30d IS NULL THEN 1 ELSE 0 END
      ) AS null_score,
      ROW_NUMBER() OVER (
        PARTITION BY unique_customer_identifier, snapshot_date
        ORDER BY
          null_score ASC,
          calls_30d DESC NULLS LAST,
          sum_download_30d DESC NULLS LAST,
          sum_upload_30d DESC NULLS LAST
      ) AS rn
    FROM read_parquet('{modeling_path}')
  )
  WHERE rn = 1
) TO '{dedup_path}' (FORMAT PARQUET);
""")

print("Saved:", dedup_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\data\interim\modeling_table_dedup.parquet


Check uniqueness

In [25]:
dedup_check = con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_unique_keys,
  COUNT(*) - COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_duplicates
FROM read_parquet('{dedup_path}')
""").fetchdf()

dedup_check

,n_rows,n_unique_keys,n_duplicates
0,3532720,3532720,0


Creating cleaned modeling table with COALESCE - removing null

In [32]:
modeling_dedup_path = str(INTERIM_DIR / "modeling_table_dedup.parquet")
modeling_clean_path = str(INTERIM_DIR / "modeling_table_clean.parquet")

con.execute(f"""
COPY (
SELECT
  unique_customer_identifier,
  snapshot_date,
  contract_status,
  contract_dd_cancels,
  dd_cancel_60_day,
  ooc_days,
  technology,
  speed,
  line_speed,
  sales_channel,
  crm_package_name,
  tenure_days,
  target_30d,

  -- Calls: if no calls then 0
  COALESCE(calls_30d, 0) AS calls_30d,
  COALESCE(avg_talk_time_30d, 0) AS avg_talk_time_30d,
  COALESCE(avg_hold_time_30d, 0) AS avg_hold_time_30d,
  COALESCE(loyalty_calls_30d, 0) AS loyalty_calls_30d,
  COALESCE(tech_calls_30d, 0) AS tech_calls_30d,
  COALESCE(billing_calls_30d, 0) AS billing_calls_30d,
  COALESCE(unknown_calls_30d, 0) AS unknown_calls_30d,

  -- Usage: if missing then 0 (no usage observed in window)
  COALESCE(avg_download_30d, 0) AS avg_download_30d,
  COALESCE(avg_upload_30d, 0) AS avg_upload_30d,
  COALESCE(sum_download_30d, 0) AS sum_download_30d,
  COALESCE(sum_upload_30d, 0) AS sum_upload_30d

FROM read_parquet('{modeling_dedup_path}')
) TO '{modeling_clean_path}' (FORMAT PARQUET);
""")

print("Saved:", modeling_clean_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\data\interim\modeling_table_clean.parquet


Build train/val/test

In [34]:
train_path = str(INTERIM_DIR / "train.parquet")
val_path   = str(INTERIM_DIR / "val.parquet")
test_path  = str(INTERIM_DIR / "test.parquet")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{modeling_clean_path}')
  WHERE snapshot_date <= DATE '2024-03-01'
) TO '{train_path}' (FORMAT PARQUET);
""")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{modeling_clean_path}')
  WHERE snapshot_date >= DATE '2024-04-01' AND snapshot_date <= DATE '2024-06-01'
) TO '{val_path}' (FORMAT PARQUET);
""")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{modeling_clean_path}')
  WHERE snapshot_date >= DATE '2024-07-01'
) TO '{test_path}' (FORMAT PARQUET);
""")

print("Built cleaned splits.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built cleaned splits.


Split stats

In [35]:
split_stats = con.execute(f"""
WITH t AS (
  SELECT 'train' AS split, * FROM read_parquet('{train_path}')
  UNION ALL
  SELECT 'val'   AS split, * FROM read_parquet('{val_path}')
  UNION ALL
  SELECT 'test'  AS split, * FROM read_parquet('{test_path}')
)
SELECT split,
       COUNT(*) AS n_rows,
       AVG(target_30d) AS target_rate,
       MIN(snapshot_date) AS min_date,
       MAX(snapshot_date) AS max_date
FROM t
GROUP BY split
ORDER BY split
""").fetchdf()

split_stats

,split,n_rows,target_rate,min_date,max_date
0,test,261737,0.034290,2024-07-01,2024-09-01
1,train,2974001,0.038784,2022-08-01,2024-03-01
2,val,296982,0.047202,2024-04-01,2024-06-01


Loading splits into pandas

In [36]:
train_df = pd.read_parquet(INTERIM_DIR / "train.parquet")
val_df   = pd.read_parquet(INTERIM_DIR / "val.parquet")
test_df  = pd.read_parquet(INTERIM_DIR / "test.parquet")

ID_COL = "unique_customer_identifier"
DATE_COL = "snapshot_date"

print("Train duplicates:", train_df.duplicated([ID_COL, DATE_COL]).sum())
print("Val duplicates  :", val_df.duplicated([ID_COL, DATE_COL]).sum())
print("Test duplicates :", test_df.duplicated([ID_COL, DATE_COL]).sum())

train_df.shape, val_df.shape, test_df.shape

Train duplicates: 0
Val duplicates  : 0
Test duplicates : 0


((2974001, 24), (296982, 24), (261737, 24))

Identify columns and missing data

In [37]:
TARGET = "target_30d"
ID_COL = "unique_customer_identifier"
DATE_COL = "snapshot_date"

feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, DATE_COL]]

# Categorical vs numeric
cat_cols = train_df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print("n_features:", len(feature_cols))
print("n_categorical:", len(cat_cols), cat_cols)
print("n_numeric:", len(num_cols))

n_features: 21
n_categorical: 4 ['contract_status', 'technology', 'sales_channel', 'crm_package_name']
n_numeric: 17


In [39]:
feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, DATE_COL]]
miss = (train_df[feature_cols].isna().mean() * 100).sort_values(ascending=False)
miss.head(10)

ooc_days               0.513887
contract_status        0.000000
contract_dd_cancels    0.000000
dd_cancel_60_day       0.000000
technology             0.000000
speed                  0.000000
line_speed             0.000000
sales_channel          0.000000
crm_package_name       0.000000
tenure_days            0.000000
dtype: float64

Adding log1p transforms

In [40]:
def add_log_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["sum_download_30d", "sum_upload_30d", "avg_talk_time_30d", "avg_hold_time_30d"]:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))
    return df

train_df = add_log_features(train_df)
val_df   = add_log_features(val_df)
test_df  = add_log_features(test_df)

In [41]:
train_df[["sum_download_30d","log1p_sum_download_30d"]].describe()

,sum_download_30d,log1p_sum_download_30d
count,2.974001e+06,2.974001e+06
mean,2.788844e+05,1.032822e+01
std,3.619388e+05,4.197321e+00
min,0.000000e+00,0.000000e+00
25%,2.064077e+04,9.935072e+00
50%,1.556415e+05,1.195532e+01
75%,4.050734e+05,1.291183e+01
max,1.751441e+07,1.667853e+01


Building X/y and feature lists

In [42]:
TARGET = "target_30d"
ID_COL = "unique_customer_identifier"
DATE_COL = "snapshot_date"

feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, DATE_COL]]

X_train, y_train = train_df[feature_cols], train_df[TARGET].astype(int)
X_val, y_val     = val_df[feature_cols], val_df[TARGET].astype(int)
X_test, y_test   = test_df[feature_cols], test_df[TARGET].astype(int)

# detect categoricals
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print("n_features:", len(feature_cols))
print("categoricals:", cat_cols)
print("n_numeric:", len(num_cols))
print("train target rate:", y_train.mean())

n_features: 25
categoricals: ['contract_status', 'technology', 'sales_channel', 'crm_package_name']
n_numeric: 21
train target rate: 0.038784452325335464


Train Model : Logistic Regression (baseline)

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),  # important!
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

prep_lr = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

lr_model_scaled = Pipeline([
    ("prep", prep_lr),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs"  # fine
    ))
])



In [45]:
from sklearn.metrics import roc_auc_score, average_precision_score

lr_model_scaled.fit(X_train, y_train)
val_proba_lr_scaled = lr_model_scaled.predict_proba(X_val)[:, 1]

print("LR (scaled) Val ROC-AUC:", roc_auc_score(y_val, val_proba_lr_scaled))
print("LR (scaled) Val PR-AUC :", average_precision_score(y_val, val_proba_lr_scaled))

LR (scaled) Val ROC-AUC: 0.9012571997647009
LR (scaled) Val PR-AUC : 0.347223029845504


Leakage checks

In [46]:
bad = [c for c in feature_cols if c in [TARGET, ID_COL, DATE_COL]]
print("Bad cols present in features:", bad)

Bad cols present in features: []


In [47]:
num_check = train_df[num_cols + [TARGET]].corr(numeric_only=True)[TARGET].abs().sort_values(ascending=False)
num_check.head(10)

target_30d                 1.000000
contract_dd_cancels        0.308821
dd_cancel_60_day           0.305296
loyalty_calls_30d          0.119174
log1p_avg_talk_time_30d    0.113375
log1p_sum_download_30d     0.111765
log1p_avg_hold_time_30d    0.110175
log1p_sum_upload_30d       0.103922
calls_30d                  0.094065
avg_hold_time_30d          0.088016
Name: target_30d, dtype: float64

In [48]:
print("train date range:", train_df["snapshot_date"].min(), "->", train_df["snapshot_date"].max())
print("val date range  :", val_df["snapshot_date"].min(), "->", val_df["snapshot_date"].max())
print("test date range :", test_df["snapshot_date"].min(), "->", test_df["snapshot_date"].max())

train date range: 2022-08-01 -> 2024-03-01
val date range  : 2024-04-01 -> 2024-06-01
test date range : 2024-07-01 -> 2024-09-01


Train HGB

In [49]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_pipe_hgb = Pipeline([("imputer", SimpleImputer(strategy="median"))])

categorical_pipe_hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

prep_hgb = ColumnTransformer([
    ("num", numeric_pipe_hgb, num_cols),
    ("cat", categorical_pipe_hgb, cat_cols),
])

hgb_model = Pipeline([
    ("prep", prep_hgb),
    ("model", HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.08,
        max_iter=300,
        random_state=42
    ))
])



In [50]:
sw_train = compute_sample_weight(class_weight="balanced", y=y_train)
hgb_model.fit(X_train, y_train, model__sample_weight=sw_train)

val_proba_hgb = hgb_model.predict_proba(X_val)[:, 1]
print("HGB Val ROC-AUC:", roc_auc_score(y_val, val_proba_hgb))
print("HGB Val PR-AUC :", average_precision_score(y_val, val_proba_hgb))

HGB Val ROC-AUC: 0.9185732510129523
HGB Val PR-AUC : 0.40219219735995887


In [51]:
import json, joblib
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_file = MODELS_DIR / "cease_risk_model.joblib"
meta_file  = MODELS_DIR / "metadata.json"

joblib.dump(hgb_model, model_file)

metadata = {
    "trained_at": datetime.utcnow().isoformat(),
    "model_name": "HistGradientBoostingClassifier",
    "lookback_days": 30,
    "horizon_days": 30,
    "val_roc_auc": float(0.9185732510129523),
    "val_pr_auc": float(0.40219219735995887),
    "features": feature_cols,
    "categorical_features": cat_cols,
    "numeric_features": num_cols,
}
with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", model_file)
print("Saved:", meta_file)

Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\cease_risk_model.joblib
Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\metadata.json


In [52]:
import os
print("Model size (MB):", round(os.path.getsize(model_file)/1e6, 2))

Model size (MB): 0.92
